# Nível 1 — Dados e primeira análise com LLM

Triagem de PLD sobre `dados/dados_nivel_1.json` (20 operações, 6 clientes), em duas partes:

- **Parte A — pandas:** diagnóstico e limpeza dos dados, normalização para BRL, agregações e duas regras determinísticas, com validação.
- **Parte B — LLM:** parecer estruturado para um cliente sinalizado, com validação de schema, medição de tokens/latência e comparação de dois prompts.

> **Princípio que rege o notebook:** somar, contar, tirar mediana e comparar com limite é **cálculo — feito em pandas**. A LLM recebe os números prontos e entra **apenas para interpretar e redigir**.

## Parte A — Tratamento e regras

### A.1 Carga e diagnóstico de qualidade

O enunciado avisa que os dados vêm de um sistema legado e não estão limpos. Antes de tratar, **medir**: procuro registros duplicados (linha inteira e `id` repetido com conteúdo divergente), datas nulas, valores inválidos e moedas misturadas — os defeitos clássicos de extração legada.

In [1]:
import json

import pandas as pd

with open("../dados/dados_nivel_1.json", encoding="utf-8") as f:
    raw = json.load(f)

TAXA_USD_BRL = raw["taxa_cambio_usd_brl"]
df = pd.DataFrame(raw["operacoes"])

print(f"{len(df)} operações | {df['cliente_id'].nunique()} clientes | taxa USD→BRL fixa = {TAXA_USD_BRL}")
df.head()

20 operações | 6 clientes | taxa USD→BRL fixa = 5.4


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [2]:
diagnostico = pd.Series({
    "registros duplicados (linha inteira idêntica)": df.duplicated().sum(),
    "mesmo id com conteúdo divergente": (df.duplicated("id", keep=False) & ~df.duplicated(keep=False)).sum(),
    "datas nulas": df["data"].isna().sum(),
    "valores nulos ou <= 0": (df["valor"].isna() | (df["valor"] <= 0)).sum(),
    "operações em moeda estrangeira": (df["moeda"] != "BRL").sum(),
}, name="ocorrências").to_frame()
diagnostico

,ocorrências
registros duplicados (linha inteira idêntica),1
mesmo id com conteúdo divergente,0
datas nulas,1
valores nulos ou <= 0,0
operações em moeda estrangeira,1


In [3]:
print("Registro duplicado (OP-0007 aparece duas vezes, idêntica em todos os campos):")
display(df[df.duplicated(keep=False)])

print("Data nula:")
display(df[df["data"].isna()])

print("Moeda estrangeira:")
display(df[df["moeda"] != "BRL"])

Registro duplicado (OP-0007 aparece duas vezes, idêntica em todos os campos):


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


Data nula:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


Moeda estrangeira:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional


### A.2 Decisões de limpeza

Três problemas plantados, três decisões (o raciocínio completo está em `docs/DECISOES.md`):

| Problema | Decisão | Justificativa |
|---|---|---|
| **OP-0007 duplicada** (2× idêntica) | Remover a cópia | Mesmo `id` e todos os campos iguais → reprocessamento do legado, não duas operações reais. Manter a cópia **muda o resultado da Regra 1** (demonstro na validação, A.5). |
| **OP-0017 sem data** ("data nao capturada pelo sistema") | Manter nos volumes; excluir **só** das regras que dependem de data | A operação aconteceu — R$ 4.300 de depósito **em espécie** (justamente o canal mais sensível em PLD) devem contar no volume. Só a data é desconhecida. |
| **OP-0013 em USD** (US$ 12.000) | Converter com a taxa fixa do arquivo (5.4) em coluna nova `valor_brl` | Comparar limites exige moeda única. Preservo `valor`/`moeda` originais para auditoria. Sem conversão, essa operação escaparia da Regra 2. |

In [4]:
df_limpo = df.drop_duplicates().copy()

# valor_brl: moeda unica para todas as comparacoes; original preservado para auditoria
df_limpo["valor_brl"] = df_limpo["valor"].where(df_limpo["moeda"] == "BRL", df_limpo["valor"] * TAXA_USD_BRL)
df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")

print(f"{len(df)} operações → {len(df_limpo)} após remover duplicata exata")
df_limpo.loc[df_limpo["moeda"] == "USD", ["id", "cliente_id", "valor", "moeda", "valor_brl"]]

20 operações → 19 após remover duplicata exata


C:\Users\usuario\AppData\Local\Temp\ipykernel_23160\3243769690.py:4: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_limpo["valor_brl"] = df_limpo["valor"].where(df_limpo["moeda"] == "BRL", df_limpo["valor"] * TAXA_USD_BRL)


,id,cliente_id,valor,moeda,valor_brl
13,OP-0013,CLI-A-4,12000,USD,64800


### A.3 Agregações

In [5]:
volume_por_cliente = (
    df_limpo.groupby("cliente_id")
    .agg(qtd_operacoes=("id", "size"), volume_total_brl=("valor_brl", "sum"))
    .sort_values("volume_total_brl", ascending=False)
)
volume_por_cliente

,qtd_operacoes,volume_total_brl
cliente_id,,
CLI-A-4,4,79500
CLI-A-1,4,57500
CLI-A-2,2,52900
CLI-A-3,3,48500
CLI-A-5,4,16900
CLI-A-6,2,10200


In [6]:
operacoes_por_canal = df_limpo["canal"].value_counts().rename_axis("canal").to_frame("qtd_operacoes")
operacoes_por_canal

,qtd_operacoes
canal,
pix,8
ted,5
boleto,3
cartao,2
especie,1


### A.4 Regras determinísticas

**Regra 1 — Fracionamento (*smurfing*):** cliente com **3+ operações na mesma data** somando **mais de R$ 50.000**, sem que **nenhuma** atinja R$ 20.000 — o padrão de quebrar um valor grande em vários pequenos para escapar de limites de reporte.

**Regra 2 — Valor atípico:** operação **acima de 5× a mediana** das operações do próprio cliente (em BRL), aplicada apenas a clientes com **4+ operações** (mediana de poucas operações não é referência).

Interpretações registradas em `DECISOES.md`: a mediana inclui a própria operação avaliada (no Nível 1 o resultado é idêntico ao excluí-la), e operações sem data ficam fora apenas da Regra 1 — seguem contando para volume e mediana.

As flags entram **no DataFrame**: `flag_fracionamento` marca cada operação do grupo (cliente, dia) enquadrado; `flag_valor_atipico` marca a operação individual que dispara a Regra 2.

In [7]:
LIMIAR_SOMA_DIA = 50_000       # soma diaria que caracteriza fracionamento
LIMIAR_OP_ISOLADA = 20_000     # nenhuma operacao isolada atinge este valor
MIN_OPS_DIA = 3                # minimo de operacoes no mesmo dia
FATOR_ATIPICO = 5              # multiplo da mediana do cliente
MIN_OPS_CLIENTE = 4            # minimo de operacoes para a Regra 2


def aplicar_regra_fracionamento(ops: pd.DataFrame) -> pd.DataFrame:
    """Regra 1: marca operações de grupos (cliente, dia) com 3+ ops, soma > 50k e todas < 20k."""
    grupos = (
        ops.dropna(subset=["data"])
        .groupby(["cliente_id", "data"])["valor_brl"]
        .agg(["size", "sum", "max"])
    )
    dias_suspeitos = grupos[
        (grupos["size"] >= MIN_OPS_DIA)
        & (grupos["sum"] > LIMIAR_SOMA_DIA)
        & (grupos["max"] < LIMIAR_OP_ISOLADA)
    ].index
    flag = ops.set_index(["cliente_id", "data"]).index.isin(dias_suspeitos)
    return ops.assign(flag_fracionamento=flag)


def aplicar_regra_valor_atipico(ops: pd.DataFrame) -> pd.DataFrame:
    """Regra 2: marca a operação acima de 5x a mediana do cliente (clientes com 4+ ops)."""
    qtd_ops_cliente = ops.groupby("cliente_id")["id"].transform("size")
    mediana_cliente = ops.groupby("cliente_id")["valor_brl"].transform("median")
    flag = (qtd_ops_cliente >= MIN_OPS_CLIENTE) & (ops["valor_brl"] > FATOR_ATIPICO * mediana_cliente)
    return ops.assign(mediana_cliente=mediana_cliente, flag_valor_atipico=flag)


df_limpo = aplicar_regra_valor_atipico(aplicar_regra_fracionamento(df_limpo))

print("Operações sinalizadas pela Regra 1 (fracionamento):")
display(df_limpo.loc[df_limpo["flag_fracionamento"], ["id", "cliente_id", "data", "valor_brl", "canal", "contraparte"]])

print("Operações sinalizadas pela Regra 2 (valor atípico):")
display(df_limpo.loc[df_limpo["flag_valor_atipico"], ["id", "cliente_id", "valor", "moeda", "valor_brl", "mediana_cliente"]])

Operações sinalizadas pela Regra 1 (fracionamento):

,id,cliente_id,data,valor_brl,canal,contraparte
0,OP-0001,CLI-A-1,2026-03-09,18100,pix,Alfa Comercio LTDA
1,OP-0002,CLI-A-1,2026-03-09,17300,pix,Alfa Comercio LTDA
2,OP-0003,CLI-A-1,2026-03-09,18800,ted,Beta Servicos ME


Operações sinalizadas pela Regra 2 (valor atípico):


,id,cliente_id,valor,moeda,valor_brl,mediana_cliente
13,OP-0013,CLI-A-4,12000,USD,64800,5450.0


In [8]:
resumo_clientes = df_limpo.groupby("cliente_id").agg(
    operacoes=("id", "size"),
    volume_brl=("valor_brl", "sum"),
    fracionamento=("flag_fracionamento", "any"),
    valor_atipico=("flag_valor_atipico", "any"),
)
resumo_clientes["sinalizado"] = resumo_clientes["fracionamento"] | resumo_clientes["valor_atipico"]
resumo_clientes

,operacoes,volume_brl,fracionamento,valor_atipico,sinalizado
cliente_id,,,,,
CLI-A-1,4,57500,True,False,True
CLI-A-2,2,52900,False,False,False
CLI-A-3,3,48500,False,False,False
CLI-A-4,4,79500,False,True,True
CLI-A-5,4,16900,False,False,False
CLI-A-6,2,10200,False,False,False


### A.5 Validação da Regra 1

Validação em dois atos:

1. **Captura quem deve e poupa o caso parecido.** CLI-A-1 e CLI-A-3 têm o mesmo desenho — 3 transferências no mesmo dia, todas abaixo de R$ 20 mil. A diferença é a soma: CLI-A-1 ultrapassa R$ 50 mil (54.200) e é sinalizado; CLI-A-3 fica abaixo (48.500) e **não** é.
2. **A limpeza importa.** Se a duplicata OP-0007 não fosse removida, o CLI-A-3 "somaria" R$ 65.700 e viraria um **falso positivo** — a regra certa sobre o dado errado produz a resposta errada.

In [9]:
validacao = (
    df_limpo.dropna(subset=["data"])
    .groupby(["cliente_id", "data"])["valor_brl"]
    .agg(qtd_ops="size", soma_dia="sum", maior_op="max")
    .query("qtd_ops >= @MIN_OPS_DIA")
    .assign(
        soma_ultrapassa_50k=lambda g: g["soma_dia"] > LIMIAR_SOMA_DIA,
        todas_abaixo_20k=lambda g: g["maior_op"] < LIMIAR_OP_ISOLADA,
    )
)
validacao["enquadra_regra_1"] = validacao["soma_ultrapassa_50k"] & validacao["todas_abaixo_20k"]
validacao

,,qtd_ops,soma_dia,maior_op,soma_ultrapassa_50k,todas_abaixo_20k,enquadra_regra_1
cliente_id,data,,,,,,
CLI-A-1,2026-03-09,3,54200,18800,True,True,True
CLI-A-3,2026-03-05,3,48500,17200,False,True,False


In [10]:
# Contrafactual: a mesma regra aplicada ao dado SUJO (duplicata mantida)
df_sujo = df.copy()
df_sujo["valor_brl"] = df_sujo["valor"].where(df_sujo["moeda"] == "BRL", df_sujo["valor"] * TAXA_USD_BRL)
df_sujo["data"] = pd.to_datetime(df_sujo["data"], errors="coerce")

flags_sujas = aplicar_regra_fracionamento(df_sujo)
comparativo = pd.DataFrame({
    "dado sujo (com duplicata)": flags_sujas.loc[flags_sujas["flag_fracionamento"]]
        .groupby("cliente_id")["valor_brl"].sum(),
    "dado limpo": df_limpo.loc[df_limpo["flag_fracionamento"]]
        .groupby("cliente_id")["valor_brl"].sum(),
})
print("Soma diária dos clientes sinalizados pela Regra 1 em cada cenário (NaN = não sinalizado):")
comparativo

Soma diária dos clientes sinalizados pela Regra 1 em cada cenário (NaN = não sinalizado):


C:\Users\usuario\AppData\Local\Temp\ipykernel_23160\1901760420.py:3: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sujo["valor_brl"] = df_sujo["valor"].where(df_sujo["moeda"] == "BRL", df_sujo["valor"] * TAXA_USD_BRL)


,dado sujo (com duplicata),dado limpo
cliente_id,,
CLI-A-1,54200,54200.0
CLI-A-3,65700,NaN


### Síntese da Parte A

| Cliente | Situação | Evidência |
|---|---|---|
| **CLI-A-1** | 🚩 **Fracionamento** (Regra 1) | 3 transferências em 09/03 para 2 contrapartes, somando R$ 54.200 — todas entre R$ 17,3 mil e R$ 18,8 mil, logo abaixo do limite de R$ 20 mil |
| **CLI-A-4** | 🚩 **Valor atípico** (Regra 2) | OP-0013: remessa internacional de US$ 12.000 (R$ 64.800) = **11,9×** a mediana do cliente (R$ 5.450) — só detectável após a conversão cambial |
| CLI-A-3 | ✅ Falso positivo **evitado** | Padrão parecido com CLI-A-1, mas soma R$ 48.500 (< 50 mil); com a duplicata OP-0007, seria sinalizado indevidamente |
| CLI-A-2, CLI-A-5, CLI-A-6 | ✅ Sem sinalização | Nenhuma regra disparada |

A Parte B toma o **CLI-A-1** — o caso de fracionamento — e pede à LLM o parecer de risco, com os números calculados aqui.